# Sistema recomanador MovieLens

Demostració completa utilitzant les funcions modulars de `src/`.


In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "src").is_dir():
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src import (
    BiasedMatrixFactorization,
    dataset_statistics,
    evaluate_model,
    get_user_history,
    load_data,
    recommend_movies,
    temporal_train_test_split,
)

DATA_DIR = PROJECT_ROOT / "data"

## 1. Càrrega de dades


In [ ]:
ratings_dataset, movies_dataset = load_data(DATA_DIR)

print("Primeres files de movies.csv:")
display(movies_dataset.head())
print("Primeres files de ratings.csv:")
display(ratings_dataset.head())

## 2. Estadístiques del dataset


In [ ]:
statistics = dataset_statistics(ratings_dataset, movies_dataset)
display(pd.DataFrame([statistics]))

## 3. Divisió train/test


In [ ]:
ratings_train_set, ratings_test_set = temporal_train_test_split(ratings_dataset)

print(f"Mida del train: {len(ratings_train_set)} valoracions")
print(f"Mida del test: {len(ratings_test_set)} valoracions")

rating_distribution = pd.concat(
    [
        ratings_train_set["rating"].value_counts().rename("train"),
        ratings_test_set["rating"].value_counts().rename("test"),
    ],
    axis=1,
).fillna(0).astype(int).sort_index()
display(rating_distribution)

## 4. Entrenament

S’utilitzen els hiperparàmetres per defecte; es mostra el MSE de train a cada època.


In [ ]:
model = BiasedMatrixFactorization()
train_mse_list = model.fit(ratings_train_set)

## 5. Corba d’aprenentatge


In [ ]:
plt.plot(range(1, len(train_mse_list) + 1), train_mse_list, marker="o")
plt.title("MSE de train per època")
plt.xlabel("Època")
plt.ylabel("MSE")
plt.xticks(range(1, len(train_mse_list) + 1))
plt.grid(True)
plt.show()

## 6. Avaluació sobre el test


In [ ]:
evaluation_metrics = evaluate_model(model, ratings_test_set, k=10)
print(f"RMSE: {evaluation_metrics['rmse']:.4f}")
print(f"MAE: {evaluation_metrics['mae']:.4f}")
print(f"NDCG@10: {evaluation_metrics['ndcg_at_k']:.4f}")
print(
    f"Prediccions vàlides: {evaluation_metrics['n_valid']} / "
    f"{len(ratings_test_set)}"
)

## 7. Exemples de predicció

Valor real i predit per a cinc parelles del test predictibles pel model.


In [ ]:
valid_test_examples = ratings_test_set.loc[
    ratings_test_set["userId"].isin(model.user2idx)
    & ratings_test_set["movieId"].isin(model.movie2idx)
].head(5).copy()
valid_test_examples["predicted_rating"] = [
    model.predict_rating(row.userId, row.movieId)
    for row in valid_test_examples.itertuples(index=False)
]
display(
    valid_test_examples[
        ["userId", "movieId", "rating", "predicted_rating"]
    ].rename(columns={"rating": "valor_real", "predicted_rating": "valor_predit"})
)

## 8. Top-10 recomanacions


In [ ]:
recommendation_user_ids = ratings_train_set["userId"].drop_duplicates().head(3)
for user_id in recommendation_user_ids:
    print(f"Recomanacions per a l’usuari {user_id}")
    display(
        recommend_movies(
            model, user_id, movies_dataset, ratings_train_set, n=10
        ).rename(columns={"predicted_rating": "valor_predit"})
    )

## 9. Historial d’un usuari


In [ ]:
history_user_id = ratings_train_set["userId"].iloc[0]
print(f"Historial de l’usuari {history_user_id}")
display(get_user_history(history_user_id, ratings_train_set, movies_dataset))